# 02 — Feature Engineering

**Goal:** Transform raw OHLCV data into features that capture different aspects of market behavior.

Each feature targets a specific anomaly signal:

| Feature Group | What It Captures | Anomaly Signal |
|---|---|---|
| Returns, Log-returns | Price changes | Extreme moves |
| Volatility | Rolling standard deviation | Volatility regime shifts |
| RSI | Momentum | Overbought/oversold extremes |
| Bollinger Bands | Price relative to moving range | Breakouts |
| MACD | Trend momentum | Trend reversals |
| ATR | True price range | Expanding/contracting ranges |
| Volume Z-score | Volume relative to recent average | Unusual trading activity |
| Rolling statistics | Higher moments of returns | Distributional shifts |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import DataFetcher
from src.data.feature_engineer import FeatureEngineer
from src.utils.helpers import load_config

plt.style.use('seaborn-v0_8-darkgrid')
config = load_config('../config/config.yaml')

fetcher = DataFetcher(config)
df = fetcher.fetch(ticker='SPY')
fe = FeatureEngineer(config)

print(f"Raw data: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 1. Run the Full Pipeline

Apply all feature engineering steps at once, then examine what we get.

In [ ]:
featured = fe.engineer(df)
print(f"After feature engineering: {featured.shape}")
print(f"\nNew columns ({featured.shape[1] - df.shape[1]} added):")
new_cols = [c for c in featured.columns if c not in df.columns]
for c in new_cols:
    print(f"  {c:20s}  min={featured[c].min():>10.4f}  max={featured[c].max():>10.4f}")

print(f"\nRows dropped (NaN warmup): {len(df) - len(featured)}")
print(f"Any remaining NaNs: {featured.isna().any().any()}")

## 2. Visualize Key Features

Plot each feature group over time alongside price to build intuition for what they capture.

In [ ]:
# Feature dashboard: price + 4 key indicators
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    row_heights=[0.3, 0.175, 0.175, 0.175, 0.175],
                    subplot_titles=['Close Price', 'RSI', 'Bollinger %B',
                                    'MACD Histogram', 'Volume Z-Score'])

# Price
fig.add_trace(go.Scatter(x=featured.index, y=featured['Close'],
              line=dict(color='#2196F3', width=1)), row=1, col=1)

# RSI with overbought/oversold zones
fig.add_trace(go.Scatter(x=featured.index, y=featured['rsi'],
              line=dict(color='#9C27B0', width=1)), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

# Bollinger %B
fig.add_trace(go.Scatter(x=featured.index, y=featured['bb_pct'],
              line=dict(color='#FF9800', width=1)), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='green', row=3, col=1)

# MACD Histogram
colors = ['#4CAF50' if v >= 0 else '#F44336' for v in featured['macd_hist']]
fig.add_trace(go.Bar(x=featured.index, y=featured['macd_hist'],
              marker_color=colors), row=4, col=1)

# Volume Z-Score
fig.add_trace(go.Scatter(x=featured.index, y=featured['volume_zscore'],
              line=dict(color='#00BCD4', width=1)), row=5, col=1)
fig.add_hline(y=3, line_dash='dash', line_color='red', row=5, col=1)
fig.add_hline(y=-3, line_dash='dash', line_color='red', row=5, col=1)

fig.update_layout(height=900, template='plotly_dark', showlegend=False,
                  title='SPY — Engineered Features Over Time')
fig.show()

## 3. Feature Correlation Matrix

Check how the engineered features correlate with each other. Highly correlated features may be redundant for some models (Isolation Forest is robust to this, linear models are not).

In [ ]:
feature_cols = fe.get_feature_columns()
available = [c for c in feature_cols if c in featured.columns]
corr = featured[available].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    text=np.round(corr.values, 2), texttemplate='%{text}',
    colorscale='RdBu_r', zmid=0
))
fig.update_layout(title='Feature Correlation Matrix (Engineered Features)',
                  template='plotly_dark', height=550, width=650)
fig.show()

# Flag highly correlated pairs
print("Highly correlated feature pairs (|r| > 0.7):")
for i in range(len(available)):
    for j in range(i+1, len(available)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            print(f"  {available[i]:20s} <-> {available[j]:20s}  r = {r:.3f}")

## 4. Feature Behavior During Known Events

Do the features actually fire during known anomaly events? Zoom into COVID crash to check.

In [ ]:
# Zoom into COVID crash period (Feb-Apr 2020)
covid = featured['2020-01-15':'2020-04-30']

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

plots = [
    ('returns', 'Daily Returns', 'tab:blue'),
    ('volatility', 'Rolling Volatility', 'tab:orange'),
    ('rsi', 'RSI', 'tab:purple'),
    ('bb_pct', 'Bollinger %B', 'tab:green'),
    ('macd_hist', 'MACD Histogram', 'tab:red'),
    ('volume_zscore', 'Volume Z-Score', 'tab:cyan'),
    ('roll_skew', 'Rolling Skewness', 'tab:brown'),
    ('roll_kurt', 'Rolling Kurtosis', 'tab:pink'),
]

for i, (col, title, color) in enumerate(plots):
    ax = axes[i // 2, i % 2]
    ax.plot(covid.index, covid[col], color=color, linewidth=1)
    ax.set_title(title)
    ax.axvline(x=pd.Timestamp('2020-02-20'), color='red', linestyle='--', alpha=0.5, label='Crash start')
    ax.axvline(x=pd.Timestamp('2020-03-23'), color='green', linestyle='--', alpha=0.5, label='Crash bottom')
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Feature Behavior During COVID Crash (Feb-Apr 2020)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observations:")
print("- Returns show extreme values (both directions) during the crash")
print("- Volatility spikes dramatically and stays elevated")  
print("- RSI drops below 30 (oversold) and stays there")
print("- Bollinger %B goes far below 0 (price below lower band)")
print("- Volume z-score spikes — panic selling volume")
print("- Rolling kurtosis increases — distribution becomes more extreme")

## Key Takeaways

1. **12 features from 5 raw columns** — the feature pipeline extracts meaningful signals from basic OHLCV data without external data sources.

2. **Features capture different anomaly types** — returns catch price shocks, volatility catches regime changes, volume z-score catches unusual activity, rolling stats catch distributional shifts.

3. **Features do fire during known events** — COVID crash shows clear signatures across all features, validating the engineering choices.

4. **Some features are correlated** — returns and log-returns are nearly identical (expected). Isolation Forest handles this naturally; for other models, feature selection may help.

5. **Warmup period** — rolling windows need ~26 days of data before producing valid values. We lose ~20 rows at the start — acceptable for multi-year datasets.

**Next:** Feed these features into anomaly detection models.